# 🔧 Excel Assistant — Fine-Tuning Notebook
**Model:** Qwen2.5-1.5B  
**Method:** QLoRA (4-bit quantized LoRA)  
**Task:** Natural Language → JSON (15-tool Excel assistant schema)  
**Hardware:** Google Colab T4 GPU (free tier)  

---
### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your `dataset.jsonl` to Google Drive
3. Run cells top to bottom

## Cell 1 — Install dependencies

In [ ]:
# Takes ~3 minutes on first run
!pip install -q transformers==4.46.0 datasets peft trl bitsandbytes accelerate
print('✅ Dependencies installed')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── EDIT THIS PATH to where you uploaded dataset.jsonl ──
DATASET_PATH = '/content/drive/MyDrive/dataset.jsonl'

# Output: fine-tuned GGUF will be saved here
OUTPUT_DIR   = '/content/drive/MyDrive/excel_assistant_model'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Drive mounted')
print(f'📂 Dataset : {DATASET_PATH}')
print(f'📦 Output  : {OUTPUT_DIR}')

## Cell 3 — Load and inspect dataset

In [ ]:
import json
from collections import Counter

# Load dataset
examples = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(json.loads(line))

print(f'📊 Total examples: {len(examples)}')

# Check tool distribution
tool_counts = Counter()
for ex in examples:
    try:
        out = json.loads(ex['output'])
        tool_counts[out['tool']] += 1
    except:
        pass

print('\n🔧 Tool distribution:')
for tool, count in sorted(tool_counts.items()):
    bar = '█' * (count // 5)
    print(f'  {tool:<25} {count:>4}  {bar}')

# Preview first example
print('\n📝 Example 0:')
print(f'  Instruction : {examples[0]["instruction"]}')
print(f'  Output      : {examples[0]["output"]}')

## Cell 4 — Format dataset into chat template

In [ ]:
import random
from datasets import Dataset

SYSTEM_PROMPT = """You are a data analysis assistant. Convert the user's natural language request into JSON using this schema:
{
  \"tool\": \"summary | filter | sort | search | aggregate | percentage | top_n | correlation | linear_regression | conditional_calc | plot | heatmap | boxplot | find_duplicates | fill_missing | drop_column\",
  \"x_column\": \"\",
  \"y_column\": \"\",
  \"agg\": \"\",
  \"n\": 0,
  \"ascending\": false,
  \"plot\": \"\",
  \"bins\": 0,
  \"condition_column\": \"\",
  \"condition_op\": \"\",
  \"condition_value\": \"\",
  \"fill_method\": \"\",
  \"columns\": []
}
Rules:
- Return ONLY valid JSON. No explanation.
- Use empty string \"\" or 0 where not applicable.
- Response must start with { and end with }."""


def format_example(ex):
    """Format a single example into Qwen2.5 chat template."""
    return (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{ex["instruction"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{ex["output"]}<|im_end|>'
    )


# Shuffle and split 90/10 train/eval
random.shuffle(examples)
split = int(len(examples) * 0.9)
train_data = examples[:split]
eval_data  = examples[split:]

train_dataset = Dataset.from_list([{'text': format_example(ex)} for ex in train_data])
eval_dataset  = Dataset.from_list([{'text': format_example(ex)} for ex in eval_data])

print(f'✅ Train examples : {len(train_dataset)}')
print(f'✅ Eval examples  : {len(eval_dataset)}')
print(f'\n📝 Formatted example preview:')
print(train_dataset[0]['text'][:400] + '...')

## Cell 5 — Load model + tokenizer (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

# Check GPU
print(f'🖥️  GPU: {torch.cuda.get_device_name(0)}')
print(f'💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'\n⬇️  Downloading {MODEL_NAME} (~3GB, takes 3-5 min)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False

print(f'✅ Model loaded!')
print(f'💾 VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## Cell 6 — Apply LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                        # LoRA rank — higher = more capacity, more VRAM
    lora_alpha=32,               # scaling factor (usually 2x rank)
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

# Show trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'✅ LoRA applied')
print(f'🔢 Trainable params : {trainable:,} ({100 * trainable / total:.2f}% of total)')
print(f'🔢 Total params     : {total:,}')

## Cell 7 — Train!

In [ ]:
from trl import SFTTrainer, SFTConfig

CHECKPOINT_DIR = '/content/checkpoints'

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,             # 3 passes over the data
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch size = 4*4 = 16
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,             # keep only last 2 checkpoints (saves disk)
    load_best_model_at_end=True,
    report_to='none',               # disable wandb
    dataset_text_field='text',
    max_seq_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print('🚀 Starting training...')
print(f'   Epochs          : {sft_config.num_train_epochs}')
print(f'   Train examples  : {len(train_dataset)}')
print(f'   Est. time       : ~45-60 min on T4')
print()

trainer.train()
print('\n✅ Training complete!')

## Cell 8 — Quick test before export

In [ ]:
import json

def predict(instruction: str) -> str:
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{instruction}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant response
    if 'assistant' in decoded:
        response = decoded.split('assistant')[-1].strip()
    else:
        response = decoded
    return response


# Test with a few examples
test_cases = [
    'show average salary by department',
    'filter rows where age is greater than 30',
    'top 5 products by revenue',
    'plot a histogram of salaries',
    'find duplicate rows',
    'what is the correlation between experience and salary',
    'fill missing values in age column with mean',
    'show me a heatmap',
]

print('🧪 Model predictions:\n')
for q in test_cases:
    response = predict(q)
    try:
        parsed = json.loads(response)
        print(f'✅ "{q}"')
        print(f'   tool={parsed.get("tool")} | x={parsed.get("x_column")} | y={parsed.get("y_column")}\n')
    except json.JSONDecodeError:
        print(f'⚠️  "{q}"')
        print(f'   Raw: {response[:100]}\n')

## Cell 9 — Save merged model to Google Drive

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

MERGED_DIR = '/content/merged_model'

print('🔀 Merging LoRA weights into base model...')

# Reload base model in fp16 for merging (not quantized)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cpu',
    trust_remote_code=True,
)

# Merge LoRA into base
merged_model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR + '/checkpoint-best')
merged_model = merged_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'✅ Merged model saved to {MERGED_DIR}')

## Cell 10 — Convert to GGUF and save to Drive

In [ ]:
import os

GGUF_NAME = 'excel_assistant_q4.gguf'
GGUF_PATH = f'{OUTPUT_DIR}/{GGUF_NAME}'

# Install llama.cpp conversion tool
print('⬇️  Setting up llama.cpp converter...')
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp -q
!pip install -q -r /content/llama.cpp/requirements.txt

# Convert to GGUF (Q4_K_M = good quality/size balance)
print('🔄 Converting to GGUF Q4_K_M...')
!python /content/llama.cpp/convert_hf_to_gguf.py \
    {MERGED_DIR} \
    --outfile /content/{GGUF_NAME} \
    --outtype q4_k_m

# Copy to Google Drive
print('☁️  Copying to Google Drive...')
!cp /content/{GGUF_NAME} "{GGUF_PATH}"

size_mb = os.path.getsize(f'/content/{GGUF_NAME}') / 1e6
print(f'\n✅ Done!')
print(f'📦 GGUF size : {size_mb:.0f} MB')
print(f'📂 Saved to  : {GGUF_PATH}')
print(f'\n👉 Download this file from Google Drive and run in Ollama:')
print(f'   ollama create excel-assistant -f Modelfile')

## Cell 11 — Create Modelfile for Ollama
Run this after downloading the GGUF to your PC

In [ ]:
# This prints the Modelfile content
# Save it as 'Modelfile' (no extension) in the same folder as your GGUF on your PC

modelfile = '''FROM ./excel_assistant_q4.gguf

SYSTEM """
You are a data analysis assistant. Convert the user's natural language request into JSON using this schema:
{{
  "tool": "summary | filter | sort | search | aggregate | percentage | top_n | correlation | linear_regression | conditional_calc | plot | heatmap | boxplot | find_duplicates | fill_missing | drop_column",
  "x_column": "",
  "y_column": "",
  "agg": "",
  "n": 0,
  "ascending": false,
  "plot": "",
  "bins": 0,
  "condition_column": "",
  "condition_op": "",
  "condition_value": "",
  "fill_method": "",
  "columns": []
}}
Rules:
- Return ONLY valid JSON. No explanation.
- Use empty string "" or 0 where not applicable.
- Response must start with {{ and end with }}.
"""

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
'''

print(modelfile)
print('---')
print('👆 Copy the above into a file called "Modelfile" (no extension) on your PC')
print('   Put it in the same folder as excel_assistant_q4.gguf')
print('   Then run: ollama create excel-assistant -f Modelfile')
print('   Test it : ollama run excel-assistant "show average salary by department"')